# Plateau’s Problem for Catenoid

This notebook provides an implementation of the Plateau's problem, which finds a minimal surface shape that connects a set of interfaces.
<!-- More details on this example, can be found in [our paper](https://arxiv.org/abs/2402.14009), Sections 4.1 and A.2. -->

### Imports and setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import GeneralNet
from training.optimizers import GaussNewton, GaussNewtonNew
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh
from training.residuals import bind_model, r_data, r_eikonal, r_mean_curvature

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### True Surface

In [ ]:
# Parametric equations for the catenoid in polar coordinates
def catenoid_surface_polar(z, phi, c=1.0):
    x = c * torch.cosh(z / c) * torch.cos(phi)
    y = c * torch.cosh(z / c) * torch.sin(phi)
    return x, y, z

# Level set function for the catenoid
def catenoid_level_set(p, c=1.0):
    x = p[:, 0]
    y = p[:, 1]
    z = p[:, 2]
    return x**2 + y**2 - (c**2 * torch.cosh(z / c)**2)

def sample_true_surface(n_samples, c=1.0):
    z = torch.linspace(-z_max, z_max, n_samples, dtype=torch.float64)
    phi = torch.linspace(-torch.pi, torch.pi, 2*n_samples, dtype=torch.float64)    
    z, phi = torch.meshgrid(z, phi, indexing='ij')
    x, y, z = catenoid_surface_polar(z.flatten(), phi.flatten(), c)
    points_on_surface = torch.vstack([x, y, z]).T
    return points_on_surface

# Bounds and number of samples
n = 1000
phi = torch.linspace(-torch.pi, torch.pi, n, dtype=torch.float64)
z_max = 1.0
c = 1.0

# Generate boundary points for the upper and lower circles
z_constant_upper = torch.full_like(phi, z_max, dtype=torch.float64)
z_constant_lower = torch.full_like(phi, -z_max, dtype=torch.float64)
x_upper, y_upper, z_upper = catenoid_surface_polar(z_constant_upper, phi, c)
x_lower, y_lower, z_lower = catenoid_surface_polar(z_constant_lower, phi, c)
pts_upper = torch.vstack([x_upper, y_upper, z_upper]).T
pts_lower = torch.vstack([x_lower, y_lower, z_lower]).T
pts_boundary = torch.cat([pts_upper, pts_lower], dim=0)
pts_surface_true = sample_true_surface(64)


# Generate the mesh using the level set function
verts, faces = get_mesh(
    lambda x: catenoid_level_set(x, c),
    N=128, 
    device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

print("True surface:")
fig.display()

### Pretraining

In [ ]:
model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
bind_model(model)
# Generate random training points
num_pretrain_samples = 10000
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)
pts_pretrain = torch.rand(num_pretrain_samples, 3, dtype=torch.float64) * (bounds[:,1] - bounds[:,0]) + bounds[:,0]


def pretrain_loss(model, params, pts):
    inputs = pts.to(dtype=torch.float64)
    x, y = pts[:, 0], pts[:, 1]
    targets = x**2 + y**2 - 1
    preds = model(inputs).squeeze(1)
    return 0.5 * (preds - targets).square().mean()

# Pretraining loop
pretrain_optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_pretrain_iters = 1000

for i in range(num_pretrain_iters):
    pretrain_optimizer.zero_grad()
    loss = pretrain_loss(model, model.params, pts_pretrain)
    loss.backward()
    pretrain_optimizer.step()
    
    if i % 100 == 0:
        print(f"Pretrain Iter {i}: Loss= {loss.item():.6f}")

print("Pretraining completed!")

verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-3, -3, -1.5], dtype=torch.float64),
    bbox_max=torch.tensor([3, 3, 1.5], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()


### Main training loop

In [ ]:
pts_eikonal = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_eikonal = torch.cat((pts_eikonal, pts_boundary))
pts_surface = pts_boundary

# Gauss-Newton weights:
loss_weights = {"data": 1.0, "eikonal": 0.001, "mean_curvature": 1.0}
# Adam weights:
# loss_weights = {"interface": 1.0, "eikonal": 0.1, "mean_curvature": 1.0}


config = {
    "pts_data": pts_boundary,
    "pts_eikonal": pts_eikonal,
    "pts_surface": pts_surface,
    "loss_weights": loss_weights,
    "regularization": 1e-6,
}

model = model.double()

In [ ]:
model = model.double()
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}
best_loss = float('inf')
# Gauss-Newton:
optim = GaussNewtonNew(model, lr=1e-1, config=config, do_woodbury=True, do_line_search=False)
# Adam:
# optim = torch.optim.Adam(model.parameters(), lr=1e-3)
start_time = time.time()
current_time = 0
for i in (pbar:=trange(100000)):
    if current_time > 7200:
        break
    optim.zero_grad()
    
    with torch.no_grad():
        loss_data  = 0.5*r_data(params, pts_boundary, torch.zeros(pts_boundary.shape[0], dtype=torch.float64)).squeeze(1).square().mean()

        loss_eikonal  = 0.5*r_eikonal(params, pts_eikonal).squeeze(1).square().mean()
    
        if i == 500:
            loss_weights = {"data": 1.0, "eikonal": 0.0, "mean_curvature": 1.0}
            config["loss_weights"] = loss_weights
            optim.config = config
    
        pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=2)
        config["pts_surface"] = pts_surface
        optim.config = config
        loss_mean_curvature = 0.5*r_mean_curvature(params, pts_surface).squeeze(1).square().mean()
           
        loss = loss_weights["data"] * loss_data + loss_weights["eikonal"] * loss_eikonal + loss_weights["mean_curvature"] * loss_mean_curvature

        loss_metric = loss_data + loss_mean_curvature
        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric.item()
        distance_over_time[current_time] = compute_distance(model.double(), catenoid_level_set, pts_surface, pts_surface_true, np.cosh(z_max))
        chamfer_over_time[current_time] = chamfer_div(model, pts_surface_true)

        if loss_metric.item() < best_loss:
            best_loss = loss_metric.item()
            best_model_state = copy.deepcopy(model.state_dict())

        pbar.set_description(f"interface: {loss_data.item():.2e} "
                            f"eikonal: {loss_eikonal.item():.2e} "
                            f"curvature: {loss_mean_curvature.item():.2e} "
                            f"error: {chamfer_over_time[current_time]:.2e} "
                            f"{len(pts_surface)}"
                            )
    optim.step()

# Optional: Load the best model after training
# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print(f"Best model loaded with loss {best_loss}")

plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()

In [ ]:
model = model.double()
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}
best_loss = float('inf')
# Gauss-Newton:

optim = GaussNewton(model, lr=1e-1, config=config, do_line_search=False)
# Adam:
# optim = torch.optim.Adam(model.parameters(), lr=1e-3)
# eikonal weight has to be small compared to h_laplace, see StabEik
start_time = time.time()
current_time = 0
for i in (pbar:=trange(100000)):
    if current_time > 1200:
        break
    optim.zero_grad()

    loss_data  = 0.5*r_data(params, pts_boundary, torch.zeros(pts_boundary.shape[0], dtype=torch.float64)).squeeze(1).square().mean()

    loss_eikonal  = 0.5*r_eikonal(params, pts_eikonal).squeeze(1).square().mean()

    if i == 500:
        loss_weights = {"data": 1.0, "eikonal": 0.0, "mean_curvature": 1.0}
        config["loss_weights"] = loss_weights
        optim.config = config

    pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=2)
    config["pts_surface"] = pts_surface
    optim.config = config
    
    loss_mean_curvature = 0.5*r_mean_curvature(params, pts_surface).squeeze(1).square().mean()
       
    loss = loss_weights["data"] * loss_data + loss_weights["eikonal"] * loss_eikonal + loss_weights["mean_curvature"] * loss_mean_curvature

    loss.backward()
    with torch.no_grad():
        loss_metric = loss_data + loss_mean_curvature
        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric.item()
        distance_over_time[current_time] = compute_distance(model.double(), catenoid_level_set, pts_surface, pts_surface_true, 1.0)
        chamfer_over_time[current_time] = chamfer_div(model, pts_surface_true)

        if loss_metric.item() < best_loss:
            best_loss = loss_metric.item()
            best_model_state = copy.deepcopy(model.state_dict())

        pbar.set_description(f"interface: {loss_data.item():.2e} "
                            f"eikonal: {loss_eikonal.item():.2e} "
                            f"curvature: {loss_mean_curvature.item():.2e} "
                            f"error: {chamfer_over_time[current_time]:.2e} "
                            f"{len(pts_surface)}"
                            )
    optim.step()

# Optional: Load the best model after training
# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print(f"Best model loaded with loss {best_loss}")

plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()

### Visualize the result

In [ ]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)

fig.display()

In [ ]:
verts, faces = get_mesh(
    model.float(), N=256, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

model.double()
mean_curvatures = r_mean_curvature(params, torch.tensor(verts, dtype=torch.float64)).squeeze(1).abs()

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)

color_map = k3d.basic_color_maps.Jet
color_range = [0, 0.005]

fig += k3d.mesh(
    verts, faces, 
    attribute=mean_curvatures.cpu().detach().numpy().astype(np.float32),
    color_range=color_range,
    color_map=color_map,
    side='double',
    flat_shading=False
)

fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()

### LBFGS

In [ ]:
pts_surface = sample_model_surface_binsearch(model.double(), pts_boundary, bound_limit=2)

In [ ]:
# import time
# import torch
# from tqdm import trange
# import matplotlib.pyplot as plt

pts_surface = sample_model_surface_binsearch(model.double(), pts_boundary, bound_limit=2)
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}
loss_cache = {}
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)

# LBFGS Optimizer
# optim = torch.optim.LBFGS(model.parameters(), lr=1e-3, max_iter=20, tolerance_grad=1e-7, tolerance_change=1e-9)
optim = torch.optim.LBFGS(model.parameters())


start_time = time.time()
current_time = 0
for i in (pbar:=trange(10000)):
    if current_time > 1200:
        break
    if i == 500:
        loss_weights = {"data": 1.0, "eikonal": 0.0, "mean_curvature": 1.0}
        config["loss_weights"] = loss_weights
        optim.config = config    
    
    def closure():
        optim.zero_grad()
        
        loss_data  = 0.5*r_data(params, pts_boundary, torch.zeros(pts_boundary.shape[0], dtype=torch.float64)).squeeze(1).square().mean()

        loss_eikonal  = 0.5*r_eikonal(params, pts_eikonal).squeeze(1).square().mean()

        try:
            pts_surface = sample_model_surface_binsearch(model.double(), pts_boundary, bound_limit=2)
        except:
            pass
        config["pts_surface"] = pts_surface
        optim.config = config
        
        loss_mean_curvature = 0.5 * r_mean_curvature(params, pts_surface).squeeze(1).square().mean()
        
        loss = (
            loss_weights["data"] * loss_data +
            loss_weights["eikonal"] * loss_eikonal +
            loss_weights["mean_curvature"] * loss_mean_curvature
        )

        loss_cache["data"] = loss_data.item()
        loss_cache["eikonal"] = loss_eikonal.item()
        loss_cache["mean_curvature"] = loss_mean_curvature.item()
        loss_cache["total"] = loss.item()
        loss_cache["pts_surface"] = pts_surface
        
        loss.backward()
        return loss
    
    loss = optim.step(closure)
    
    with torch.no_grad():
       
        loss_metric = loss_cache["data"] + loss_cache["mean_curvature"]
        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric
        distance_over_time[current_time] = compute_distance(model.double(), catenoid_level_set, pts_surface, pts_surface_true, 1.0)
        chamfer_over_time[current_time] = chamfer_div(model, pts_surface_true)
        pbar.set_description(f"interface: {loss_cache['data']:.2e} "
                             f"eikonal: {loss_cache['eikonal']:.2e} "
                             f"curvature: {loss_cache['mean_curvature']:.2e} "
                             f"error: {chamfer_over_time[current_time]:.2e} "
                             f"{len(loss_cache['pts_surface'])}"
                            )

plt.plot(loss_over_iters.keys(), loss_over_iters.values())
plt.semilogy()
plt.show()


In [ ]:
plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()